In [0]:
# Daily weather summary for all cities
# Includes max, min, and total precipitation for each city

from pyspark.sql.functions import to_date, col, max, min, sum, round

silver_df = spark.read.table("weather_project.north_texas_weather.silver_hourly_multi_city")

# Add a date column so we can group by days and not hour (for aggregation functions)
daily_weather_summary = silver_df.withColumn("date", to_date(col("utc_time")))

daily_weather_summary = daily_weather_summary.groupBy("city", "date").agg(
    round(max("temperature_celsius"), 2).alias("max_temperature_celsius"),
    round(min("temperature_celsius"), 2).alias("min_temperature_celsius"),
    round(max("temperature_fahrenheit"), 2).alias("max_temperature_fahrenheit"),
    round(min("temperature_fahrenheit"), 2).alias("min_temperature_fahrenheit"),
    round(sum("precipitation_inches"), 2).alias("total_precipitation_inches")
)

daily_weather_summary = daily_weather_summary.orderBy("city", "date")

display(daily_weather_summary)

daily_weather_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_project.north_texas_weather.gold_daily_weather_summary")

print("Successfully built and saved the multi-city Gold Daily Weather Summary!")

In [0]:
# City aggregates for all cities
# Includes weather comparisons between each city

from pyspark.sql.functions import to_date, col, max, min, sum, round, avg

silver_df = spark.read.table("weather_project.north_texas_weather.silver_hourly_multi_city")

city_aggregates = silver_df.groupBy("city").agg(
    round(avg("temperature_celsius"), 2).alias("overall_avg_temperature_celsius"),
    round(avg("temperature_fahrenheit"), 2).alias("overall_avg_temperature_fahrenheit"),
    round(avg("precipitation_inches"), 2).alias("overall_avg_total_precipitation_inches"),
    round(max("temperature_celsius"), 2).alias("absolute_max_temperature_celsius"),
    round(min("temperature_celsius"), 2).alias("absolute_min_temperature_celsius"),
    round(max("temperature_fahrenheit"), 2).alias("absolute_max_temperature_fahrenheit"),
    round(min("temperature_fahrenheit"), 2).alias("absolute_min_temperature_fahrenheit"),
)

display(city_aggregates)

city_aggregates.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_project.north_texas_weather.gold_city_aggregates")

print("Successfully built and saved the multi-city Gold City Aggregates!")

In [0]:
# Extreme weather events

from pyspark.sql.functions import col, when, concat, lit

df_daily_weather_summary = spark.read.table("weather_project.north_texas_weather.gold_daily_weather_summary")

df_extreme_weather_events = df_daily_weather_summary.filter(
    (col("max_temperature_fahrenheit") >= 100) |
    (col("min_temperature_fahrenheit") <= 32) |
    (col("total_precipitation_inches") >= 2)
)

df_extreme_weather_events = df_extreme_weather_events \
    .withColumn("is_extreme_heat", when(col("max_temperature_fahrenheit") >= 100, True).otherwise(False)) \
    .withColumn("is_extreme_cold", when(col("min_temperature_fahrenheit") <= 32, True).otherwise(False)) \
    .withColumn("is_extreme_precip", when(col("total_precipitation_inches") >= 2.0, True).otherwise(False))

df_extreme_weather_events = df_extreme_weather_events.orderBy(col("date").desc())

display(df_extreme_weather_events)

df_extreme_weather_events.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("weather_project.north_texas_weather.gold_extreme_weather_events")

print("Successfully built and saved the Gold Extreme Weather Events!")

In [0]:
# Monthly trends

from pyspark.sql.functions import to_date, col, max, min, sum, round, month, avg, year, date_format

df_daily_weather_summary = spark.read.table("weather_project.north_texas_weather.gold_daily_weather_summary")

df_monthly_trends = df_daily_weather_summary \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", date_format(col("date"), "MMMM"))

df_monthly_trends = df_monthly_trends.groupBy("city", "year", "month").agg(
    round(avg("max_temperature_celsius"), 2).alias("avg_high_temperature_celsius"),
    round(avg("min_temperature_celsius"), 2).alias("avg_low_temperature_celsius"),
    round(avg("max_temperature_fahrenheit"), 2).alias("avg_high_temperature_fahrenheit"),
    round(avg("min_temperature_fahrenheit"), 2).alias("avg_low_temperature_fahrenheit"),
    round(sum("total_precipitation_inches"), 2).alias("total_monthly_precipitation_inches")
)

df_monthly_trends = df_monthly_trends.orderBy("city", "year")

display(df_monthly_trends)

df_monthly_trends.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("weather_project.north_texas_weather.gold_monthly_trends")

print("Successfully built and saved the Gold Monthly Trends!")